### Processing and splitting pipline

In [ ]:
#Processing
import json, math, os, random
from collections import defaultdict
from datetime import datetime, timezone
import zstandard as zstd
import pandas as pd

import gdown 


In [ ]:
os.makedirs("dataset", exist_ok=True)

FILES = {"RS_2026-01": "1gNgXqF9YuGrsvu_e8MShloUkKd_Xj_Kj"}

def download_file(file_id, output_path):
    if not os.path.exists(output_path):
        url = f"https://drive.google.com/uc?id={file_id}"
        print(f"Downloading {output_path}...")
        gdown.download(url, output_path, quiet=False)
    else:
        print(f"{output_path} already exists.")

for split, file_id in FILES.items():
    download_file(file_id, f"data/{split}.zst")

raw_data = "dataset/raw/RS_2026-01.zst"

In [ ]:
# Paths
input_path = "dataset/raw/RS_2026-01.zst"
train_path = "dataset/clean_data/train.jsonl"
val_path = "dataset/clean_data/val.jsonl"
test_path = "dataset/clean_data/test.jsonl"

os.makedirs(os.path.dirname(train_path), exist_ok=True)

RANDOM_SEED = 42
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

BIN_CAPS = {
    "zero": 150_000,
    "low": 150_000,
    "mid": 100_000,
    "high": 50_000,
}

JUNK_TITLES = {"[deleted]", "[removed]", "[ Removed by moderator ]"}

# Helpers
def get_score_bin(score: int) -> str:
    if score <= 0: return "zero"
    if score <= 10: return "low"
    if score <= 100: return "mid"
    return "high"

def is_valid_post(post: dict) -> bool:
    title = post.get("title")
    if not title or title in JUNK_TITLES:
        return False
    if post.get("score") is None or post.get("created_utc") is None:
        return False
    return True

def process_post(post: dict, dt: datetime) -> dict:
    title = post["title"]
    score = max(post["score"], 0)
    content = post.get("selftext", "") or ""
    if content in {"[deleted]", "[removed]"}:
        content = ""
    content = content.strip()

    return {
        "title": title,
        "content": content,
        "has_content": int(len(content) > 0),
        "subreddit": post["subreddit"],
        "nsfw": int(post.get("over_18", False)),
        "is_self": int(post.get("is_self", False)),
        "hour": dt.hour,
        "day_of_week": dt.weekday(),
        "is_weekend": int(dt.weekday() >= 5),
        "title_length": len(title.split()),
        "char_length": len(title),
        "uppercase_ratio": sum(1 for c in title if c.isupper()) / max(len(title), 1),
        "num_exclamations": title.count("!"),
        "num_questions": title.count("?"),
        "content_length": len(content.split()),
        "score": score,
        "log_score": math.log1p(score),
    }

def raw_stream(path: str):
    """Yields raw parsed post dicts straight from the .zst file."""
    with open(path, "rb") as fh:
        dctx = zstd.ZstdDecompressor()
        buffer = ""
        with dctx.stream_reader(fh) as reader:
            for chunk in iter(lambda: reader.read(16384), b""):
                buffer += chunk.decode("utf-8", errors="replace")
                while "\n" in buffer:
                    line, buffer = buffer.split("\n", 1)
                    try:
                        post = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    if is_valid_post(post):
                        yield post

def capped_stream(path: str):
    """Yields valid posts, stopping each bin once its cap is reached."""
    bin_seen = defaultdict(int)
    for post in raw_stream(path):
        score = max(post.get("score", 0), 0)
        bin_name = get_score_bin(score)
        if bin_seen[bin_name] >= BIN_CAPS[bin_name]:
            continue
        bin_seen[bin_name] += 1
        yield post

print("Pass 1: counting bins...")
bin_counts = defaultdict(int)

for post in capped_stream(input_path):
    score = max(post.get("score", 0), 0)
    bin_counts[get_score_bin(score)] += 1

print("bin counts:", dict(bin_counts))

bin_cutoffs = {}
for bin_name, n in bin_counts.items():
    n_train = int(n * SPLIT_RATIOS["train"])
    n_val = int(n * SPLIT_RATIOS["val"])
    bin_cutoffs[bin_name] = (n_train, n_train + n_val)
    print(f"  bin={bin_name:<5}  n={n:>7,}  "
          f"train={n_train:>6,}  val={n_val:>5,}  "
          f"test={n - n_train - n_val:>6,}")

rng = random.Random(RANDOM_SEED)
bin_assignments: dict[str, list[str]] = {}

for bin_name, n in bin_counts.items():
    n_train, n_trainval = bin_cutoffs[bin_name]
    assignments = (["train"] * n_train + ["val"] * (n_trainval - n_train) + ["test"] * (n - n_trainval))
    rng.shuffle(assignments)
    bin_assignments[bin_name] = assignments

print("\nPass 2: writing splits...")
bin_cursors = defaultdict(int)
totals = defaultdict(int)

with open(train_path, "w") as train_f, \
     open(val_path, "w") as val_f, \
     open(test_path, "w") as test_f:

    output_files = {"train": train_f, "val": val_f, "test": test_f}

    for post in capped_stream(input_path):
        dt = datetime.fromtimestamp(post["created_utc"], tz=timezone.utc)
        processed = process_post(post, dt)
        bin_name = get_score_bin(processed["score"])

        cursor = bin_cursors[bin_name]
        split_name = bin_assignments[bin_name][cursor]
        bin_cursors[bin_name] += 1

        output_files[split_name].write(json.dumps(processed) + "\n")
        totals[split_name] += 1

print("\nSPLIT TOTALS:")
for name in ("train", "val", "test"):
    print(f"{name:5s}: {totals[name]:>7,}")

print("\nPER-BIN BREAKDOWN:")
for bin_name in ("zero", "low", "mid", "high"):
    n = bin_counts[bin_name]
    print(f"{bin_name:<5}: {n:>7,} posts  "
          f"(capped at {BIN_CAPS[bin_name]:,})")

print("\nDone!")

##### Samples for TAs

In [ ]:
def create_random_subsets(input_file, output_file, sample_ratio, seed = 42):
    """
    Random sampling for large JSONL files
    """
    rng = random.Random(seed)
    with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "w", encoding="utf-8") as outfile:

        lines_written = 0
        lines_seen = 0
        for line in infile:
            lines_seen += 1
            if rng.random() < sample_ratio:
                outfile.write(line)
                lines_written += 1
            print(f"Sampled {lines_written:,} / {lines_seen:,} lines ({lines_written/lines_seen:.1%})")
    print(f"Random subset saved: {output_file}")
    print(f"Lines written: {lines_written:,}")

In [ ]:
create_random_subsets(train_path, "dataset/subset_medium/train_medium.jsonl", 0.50)
create_random_subsets(val_path,   "dataset/subset_medium/val_medium.jsonl", 0.50)
create_random_subsets(test_path,  "dataset/subset_medium/test_medium.jsonl", 0.50)

create_random_subsets(train_path, "dataset/subset_small/train_small.jsonl",0.25)
create_random_subsets(val_path,   "dataset/subset_small/val_small.jsonl", 0.25)
create_random_subsets(test_path,  "dataset/subset_small/test_small.jsonl", 0.25)